# Support Vector Machine (SVM)

Course 3: Supervised Learning, Klasifikasi

SVM mencari hyperplane (garis atau bidang pemisah) dengan **margin terbesar** antara dua kelas. Data yang paling dekat dengan hyperplane disebut support vector dan merekalah yang menentukan posisi hyperplane. Untuk data yang tidak bisa dipisahkan secara linear, SVM memakai kernel untuk bekerja di ruang fitur berdimensi lebih tinggi. Di notebook ini aku melatih SVM, mengatur `C` dan kernel, lalu menggambar hyperplane dan margin-nya.

## Ide kernel secara singkat

Kernel adalah fungsi yang menerima dua vektor dan mengembalikan satu angka, seperti dot product. Contoh kernel polinomial sederhana: $k(x, y) = (x^T y)^2$. Untuk $x = (x_1, x_2)$ dan $y = (y_1, y_2)$:

$$k(x, y) = (x_1 y_1 + x_2 y_2)^2 = x_1^2 y_1^2 + x_2^2 y_2^2 + 2 x_1 x_2 y_1 y_2$$

Ini sama dengan dot product dua vektor 3 dimensi $(x_1^2, x_2^2, \sqrt{2} x_1 x_2)$ dan $(y_1^2, y_2^2, \sqrt{2} y_1 y_2)$. Jadi SVM dengan kernel non-linear bisa bekerja seolah-olah fiturnya sudah dipetakan ke dimensi yang lebih tinggi (tempat data jadi bisa dipisah secara linear), tanpa benar-benar menghitung pemetaan itu.

## Dataset

Datanya `food_items_binary.csv`, kandungan gizi makanan untuk penderita diabetes dengan dua kelas: makanan yang sebaiknya dimakan lebih sering (`More Often`, kelas 1) atau lebih jarang (`Less Often`, kelas 0). Dibaca langsung dari link:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/food_items_binary.csv

## 1. Import dan baca data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 123

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/food_items_binary.csv"
food = pd.read_csv(URL)
print(food.shape)
food.head(10)

In [ ]:
X = food.iloc[:, :-1]
y = food.iloc[:, -1]
print("jumlah fitur:", X.shape[1])
print((y.value_counts(normalize=True) * 100).round(1).to_string())
y.value_counts().plot.bar(color=["tab:red", "tab:green"], figsize=(5, 3))
plt.ylabel("jumlah")
plt.show()

Ada dua kelas dan jumlahnya tidak seimbang: kebanyakan makanan masuk ke kelas `Less Often`. Karena itu aku pakai F1 (bukan hanya akurasi) untuk menilai model.

## 2. Pisahkan data dan siapkan evaluasi

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)

def evaluasi(y_benar, y_pred):
    return {
        "accuracy": accuracy_score(y_benar, y_pred),
        "precision": precision_score(y_benar, y_pred),
        "recall": recall_score(y_benar, y_pred),
        "f1": f1_score(y_benar, y_pred),
    }

def buat_svm(**param):
    # SVM peka terhadap skala fitur, jadi scaler dimasukkan ke dalam pipeline
    return Pipeline([("scaler", MinMaxScaler()), ("svm", SVC(**param))])

## 3. SVM dengan parameter default

In [ ]:
svm_default = buat_svm().fit(X_train, y_train)
pd.Series(evaluasi(y_test, svm_default.predict(X_test)), name="default").round(4)

Hasil defaultnya sudah cukup baik. Berikutnya kita lihat apakah bisa diperbaiki dengan mengatur `C` dan kernel.

## 4. Mengatur `C` dan kernel

- `C` adalah parameter regularisasi. `C` besar membuat model berusaha mengklasifikasikan semua data latih dengan benar (margin lebih kecil, risiko overfit). `C` kecil membiarkan beberapa kesalahan demi margin yang lebih lebar.
- `kernel` menentukan jenis pemetaan fitur: `linear`, `rbf` (Gaussian), `poly`, atau `sigmoid`.

In [ ]:
svm_c10 = buat_svm(C=10, kernel="rbf").fit(X_train, y_train)
pd.Series(evaluasi(y_test, svm_c10.predict(X_test)), name="C=10, rbf").round(4)

### Latihan: coba beberapa kombinasi C dan kernel

In [ ]:
baris = []
for kernel in ["linear", "rbf", "poly", "sigmoid"]:
    for C in [0.1, 1, 10, 100]:
        m = buat_svm(C=C, kernel=kernel).fit(X_train, y_train)
        baris.append({"kernel": kernel, "C": C, **evaluasi(y_test, m.predict(X_test))})

coba = pd.DataFrame(baris)
coba.sort_values("f1", ascending=False).round(4).head(8)

Tabel di atas diurutkan dari F1 tertinggi. Ini hanya untuk melihat gambaran pengaruh `C` dan kernel. Pemilihan yang benar dilakukan dengan cross validation di bagian berikutnya, karena memilih lewat data test membuat data test ikut menentukan model.

## 5. Grid search

In [ ]:
pipa = buat_svm()
param_grid = {
    "svm__C": [1, 10, 100],
    "svm__kernel": ["poly", "rbf", "sigmoid"],
}

grid = GridSearchCV(pipa, param_grid, scoring="f1", cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print("kombinasi terbaik:", grid.best_params_)
print("F1 cross validation:", round(grid.best_score_, 4))

In [ ]:
svm_terbaik = grid.best_estimator_
pd.DataFrame({
    "default": evaluasi(y_test, svm_default.predict(X_test)),
    "hasil grid search": evaluasi(y_test, svm_terbaik.predict(X_test)),
}).T.round(4)

Perbandingan di atas menunjukkan apakah tuning memperbaiki hasil dibanding SVM default.

## 6. Menggambar hyperplane dan margin

Hyperplane di atas tiga dimensi tidak bisa digambar, jadi aku sederhanakan datanya: hanya dua fitur (`Calories` dan `Dietary Fiber`) dan 1000 baris pertama. Kelas mayoritas di-undersample supaya kedua kelas seimbang dan gambarnya lebih jelas, lalu fitur diskalakan ke 0 sampai 1.

In [ ]:
kecil = food[["Calories", "Dietary Fiber", food.columns[-1]]].iloc[:1000]
nama_kelas = kecil.columns[-1]

# undersampling manual: ambil sampel acak dari kelas mayoritas sebanyak kelas minoritas
n_min = kecil[nama_kelas].value_counts().min()
seimbang = pd.concat([g.sample(n_min, random_state=RS) for _, g in kecil.groupby(nama_kelas)])

X2 = MinMaxScaler().fit_transform(seimbang[["Calories", "Dietary Fiber"]])
y2 = seimbang[nama_kelas].values
print("data setelah di-undersample:", X2.shape, "| per kelas:", np.bincount(y2))

In [ ]:
def gambar_hyperplane(model, judul):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    DecisionBoundaryDisplay.from_estimator(
        model, X2, response_method="decision_function", plot_method="contour",
        levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], alpha=0.6, ax=ax)
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap=plt.cm.Paired, s=25)
    ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
               s=110, facecolors="none", edgecolors="k", linewidth=1)
    ax.set(xlabel="Calories (skala 0-1)", ylabel="Dietary Fiber (skala 0-1)", title=judul)
    plt.show()
    print("jumlah support vector:", len(model.support_vectors_))

svm_linear = SVC(C=1000, kernel="linear").fit(X2, y2)
gambar_hyperplane(svm_linear, "SVM linear (C=1000)")

Garis penuh adalah hyperplane, garis putus-putus adalah batas margin, dan titik yang dilingkari adalah support vector yang menentukan posisi hyperplane.

In [ ]:
svm_rbf = SVC(C=100, kernel="rbf").fit(X2, y2)
gambar_hyperplane(svm_rbf, "SVM kernel RBF (C=100)")

Dengan kernel RBF, batas keputusannya melengkung (non-linear) mengikuti bentuk data.

### Latihan: pengaruh C dan kernel pada hyperplane

In [ ]:
svm_poly = SVC(C=100, kernel="poly").fit(X2, y2)
gambar_hyperplane(svm_poly, "SVM kernel polinomial (C=100)")

svm_rbf_kecil = SVC(C=1, kernel="rbf").fit(X2, y2)
gambar_hyperplane(svm_rbf_kecil, "SVM kernel RBF (C=1)")

Dengan `C` yang lebih kecil, margin cenderung lebih lebar dan jumlah support vector biasanya bertambah, karena model lebih toleran terhadap titik yang salah klasifikasi.

## Ringkasan

- SVM mencari hyperplane dengan margin terbesar, dan kernel memungkinkan batas keputusan yang non-linear.
- SVM peka terhadap skala fitur, jadi scaler dimasukkan ke dalam pipeline.
- `C` mengatur keseimbangan antara margin lebar dan kesalahan klasifikasi di data latih, sedangkan kernel mengatur bentuk batas keputusan.
- Kombinasi terbaik dicari dengan `GridSearchCV` di data train, dan hasil akhirnya diuji di data test.